## Read in data

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

# Provide explicit schema
schema = StructType([
    StructField("Customer_ID",       IntegerType(), False),
    StructField("Age",               IntegerType(), True),
    StructField("Gender",            StringType(),  True),
    StructField("Annual_Income",     IntegerType(),  True),
    StructField("Spending_Score",    IntegerType(), True),
    StructField("Membership_Years",  DoubleType(), True),
    StructField("Online_Purchases",  IntegerType(), True),
    StructField("Discount_Usage",    DoubleType(),  True),
    StructField("Churn",             IntegerType(), True),
])

df = spark.read.csv(
    "/Volumes/workspace/default/ecommerce_data/customer-behavior-and-churn-prediction-dataset/ecommerce_customer_data.csv",
    header="true",
    schema=schema
)
display(df)

In [0]:
# Define seed for reproducibility
RANDOM_SEED = 42

## Drop duplicates, inspect schema and nulls

In [0]:
# Inspect schema
df.printSchema()

In [0]:
# Display number of rows
df.count()

In [0]:
# Drop duplicate entries
df = df.dropDuplicates()
df.count()

In [0]:
# Count missing values in all columns
from pyspark.sql.functions import isnan, when, count, col
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

## EDA: Age, Gender, Annual_Income, Spending_Score, Membership_Years, Online_Purchases, Discount_Usage, Churn

In [0]:
# Drop Customer_ID
df = df.drop("Customer_ID")
df.show(5)

### Target variable distribution

In [0]:
# Examine Churn class balance
from pyspark.sql.functions import count, avg as spark_avg

churn_counts = df.groupBy('Churn').count().orderBy('Churn')
display(churn_counts)

# Overall churn rate
overall_churn_rate = df.select(spark_avg('Churn')).collect()[0][0]
print(f'Overall churn rate: {overall_churn_rate:.2%}')

In [0]:
# Visualise class balance using matplotlib for a reproducible, exportable chart
import pandas as pd
import matplotlib.pyplot as plt

churn_pd = churn_counts.toPandas()
churn_pd['label'] = churn_pd['Churn'].map({0: 'Retained', 1: 'Churned'})

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(churn_pd['label'], churn_pd['count'], color=['#4C72B0', '#DD8452'])
ax.bar_label(bars, fmt='%d')
ax.set_title('Churn Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(f'Class ratio (retained:churned) = {churn_pd["count"].iloc[0] / churn_pd["count"].iloc[1]:.2f}:1')
print('NOTE: If highly imbalanced, consider class weighting, SMOTE, or adjusted decision thresholds.')

### Descriptive statistics

In [0]:
# Define numeric columns (excluding Churn which is the target, and Gender which is categorical)
numeric_cols = [col_name for col_name, dtype in df.dtypes
                if dtype in ('int', 'double', 'float', 'decimal')
                and col_name != 'Churn']
print('Numeric feature columns:', numeric_cols)

In [0]:
# Examine descriptive statistics of numeric columns
display(df.select(numeric_cols).describe())

In [0]:
# Examine distributions of each numeric column using DataBricks' built-in Visualization feature
display(df.select(numeric_cols))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Examine distribution of gender using DataBricks' built-in Visualization feature
display(df.select("Gender"))

Databricks visualization. Run in Databricks to view.

### Feature skewness

In [0]:
# Compute skewness for each numeric feature — high absolute skew (>1) may warrant
# log/sqrt transformation for linear models, though tree-based models are robust to it
from pyspark.sql.functions import skewness

skew_vals = df.select([skewness(col(c)).alias(c) for c in numeric_cols])
display(skew_vals)

# Materialise for printed summary
skew_pd = skew_vals.toPandas().T.rename(columns={0: 'skewness'}).sort_values('skewness', key=abs, ascending=False)
print(skew_pd.to_string())
print('\nFeatures with |skewness| > 1 (consider transformation for linear models):')
print(skew_pd[skew_pd['skewness'].abs() > 1].index.tolist())

### Outlier detection

In [0]:
# Use IQR-based outlier detection for each numeric feature
# Outliers can distort distance-based models; tree models are largely unaffected
import seaborn as sns

quantile_levels = [0.01, 0.25, 0.50, 0.75, 0.99]
quantiles = {c: df.approxQuantile(c, quantile_levels, 0.01) for c in numeric_cols}

print(f'{'Feature':<25} {'P1':>8} {'P25':>8} {'P50':>8} {'P75':>8} {'P99':>8} {'IQR':>8} {'Lower':>10} {'Upper':>10}')
print('-' * 100)
outlier_summary = {}
for feat, qs in quantiles.items():
    p1, p25, p50, p75, p99 = qs
    iqr = p75 - p25
    lower = p25 - 1.5 * iqr
    upper = p75 + 1.5 * iqr
    outlier_summary[feat] = (lower, upper)
    print(f'{feat:<25} {p1:>8.2f} {p25:>8.2f} {p50:>8.2f} {p75:>8.2f} {p99:>8.2f} {iqr:>8.2f} {lower:>10.2f} {upper:>10.2f}')

In [0]:
# Box plots for visual outlier inspection (matplotlib)
import math

n_cols_plot = 3
n_rows_plot = math.ceil(len(numeric_cols) / n_cols_plot)
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(14, 4 * n_rows_plot))
axes = axes.flatten()

# Sample to pandas for plotting (safe on large datasets with a reasonable sample size)
sample_pd = df.select(numeric_cols).sample(fraction=0.1, seed=RANDOM_SEED).toPandas()

for i, feat in enumerate(numeric_cols):
    axes[i].boxplot(sample_pd[feat].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#4C72B0', alpha=0.6))
    axes[i].set_title(feat)
    axes[i].set_xticklabels([])

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Box Plots (10% sample)', y=1.01)
plt.tight_layout()
plt.show()

### Class-conditional churn rates per feature segment

In [0]:
# Compute computed churn rate per feature bin
# This reveals which segments genuinely drive churn vs. just being large.
from pyspark.sql.functions import avg as spark_avg, count as spark_count

from pyspark.ml.feature import Bucketizer

# Bin Age for visual analysis
df = Bucketizer(
    splits=[18, 25, 35, 45, 55, 65, 75],
    inputCol="Age",
    outputCol="Age_Binned"
).transform(df)

df = df.withColumn("Age_Binned_Label", when(df.Age_Binned == 0, "18-25")
                   .when(df.Age_Binned == 1, "25-35")
                   .when(df.Age_Binned == 2, "35-45")
                   .when(df.Age_Binned == 3, "45-55")
                   .when(df.Age_Binned == 4, "55-65")
                   .when(df.Age_Binned == 5, "65-75"))

In [0]:
# Bin Annual_Income for visual analysis
df = Bucketizer(
    splits=[20000, 40000, 60000, 80000, 100000, 120000, 140000, 160000],
    inputCol="Annual_Income",
    outputCol="Annual_Income_Binned"
).transform(df)

df = df.withColumn("Annual_Income_Binned_Label", when(df.Annual_Income_Binned == 0, "1_$20K-$40K")
                   .when(df.Annual_Income_Binned == 1, "2_$40K-$60K")
                   .when(df.Annual_Income_Binned == 2, "3_$60K-$80K")
                   .when(df.Annual_Income_Binned == 3, "4_$80K-$100K")
                   .when(df.Annual_Income_Binned == 4, "5_$100K-$120K")
                   .when(df.Annual_Income_Binned == 5, "6_$120K-$140K")
                   .when(df.Annual_Income_Binned == 6, "7_$140K-$160K"))

In [0]:
# Bin Spending_Score for visual analysis
df = Bucketizer(
    splits=[0, 25, 50, 75, 100],
    inputCol="Spending_Score",
    outputCol="Spending_Score_Binned"
).transform(df)

df = df.withColumn("Spending_Score_Binned_Label", when(df.Spending_Score_Binned == 0, "0-25")
                   .when(df.Spending_Score_Binned == 1, "25-50")
                   .when(df.Spending_Score_Binned == 2, "50-75")
                   .when(df.Spending_Score_Binned == 3, "75-100"))

In [0]:
# Bin Membership_Years for visual analysis
df = Bucketizer(
    splits=[0, 2, 4, 6, 8, 10],
    inputCol="Membership_Years",
    outputCol="Membership_Years_Binned"
).transform(df)

df = df.withColumn("Membership_Years_Binned_Label", when(df.Membership_Years_Binned == 0, "0-2 years")
                   .when(df.Membership_Years_Binned == 1, "2-4 years")
                   .when(df.Membership_Years_Binned == 2, "4-6 years")
                   .when(df.Membership_Years_Binned ==3, "6-8 years")
                   .when(df.Membership_Years_Binned == 4, "8-10 years"))

In [0]:
# Bin Online_Purchases for visual analysis
df = Bucketizer(
    splits=[0, 25, 50, 75, 100, 125, 150, 175, 200],
    inputCol="Online_Purchases",
    outputCol="Online_Purchases_Binned"
).transform(df)

df = df.withColumn("Online_Purchases_Binned_Label", when(df.Online_Purchases_Binned == 0, "1_0-25")
                   .when(df.Online_Purchases_Binned == 1, "2_25-50")
                   .when(df.Online_Purchases_Binned == 2, "3_50-75")
                   .when(df.Online_Purchases_Binned == 3, "4_75-100")
                   .when(df.Online_Purchases_Binned == 4, "5_100-125")
                   .when(df.Online_Purchases_Binned == 5, "6_125-150")
                   .when(df.Online_Purchases_Binned == 6, "7_150-175")
                   .when(df.Online_Purchases_Binned == 7, "8_175-200"))

In [0]:
# Bin Discount_Usage for visual analysis
df = Bucketizer(
    splits=[0, 0.2, 0.4, 0.6, 0.8, 1],
    inputCol="Discount_Usage",
    outputCol="Discount_Usage_Binned"
).transform(df)

df = df.withColumn("Discount_Usage_Binned_Label", when(df.Discount_Usage_Binned == 0, "0-20%")
                   .when(df.Discount_Usage_Binned == 1, "20-40%")
                   .when(df.Discount_Usage_Binned == 2, "40-60%")
                   .when(df.Discount_Usage_Binned == 3, "60-80%")
                   .when(df.Discount_Usage_Binned == 4, "80-100%"))

In [0]:
# Compute churn rate and segment size for each binned feature — gives actionable signal
# beyond just co-distributions. Sorted descending by churn_rate.
X_feats = ["Age_Binned_Label", "Annual_Income_Binned_Label", "Spending_Score_Binned_Label",
           "Membership_Years_Binned_Label", "Online_Purchases_Binned_Label",
           "Discount_Usage_Binned_Label", "Gender"]

for feat in X_feats:
    print(f'--- Churn rate by {feat} ---')
    rate_df = (
        df.groupBy(feat)
        .agg(
            spark_count('*').alias('n'),
            spark_avg('Churn').alias('churn_rate')
        )
        .orderBy('churn_rate', ascending=False)
    )
    display(rate_df)
    print()

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### Explore relationships visually

In [0]:
# Inspect bivariate relationships between features and churn using DataBricks' built-in Visualization feature
X_feats_bin = ["Age_Binned_Label", "Annual_Income_Binned_Label", "Spending_Score_Binned_Label", "Membership_Years_Binned_Label", "Online_Purchases_Binned_Label", "Discount_Usage_Binned_Label", "Gender"]

[display(df[X, "Churn"]) for X in X_feats_bin]

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Inspect multi-variate relationships between features and churn using DataBricks' built-in Visualization feature
# Extract unique combinations of binned feature columns for heat map visualizations
from itertools import combinations
unique_combos_bin = list(combinations(X_feats_bin, 2))

# Display dataframes comprising unique combos and Churn column
for combo_bin in unique_combos_bin:
    combo_bin = list(combo_bin)
    combo_bin.append('Churn')
    display(df[combo_bin])

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

...

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### Correlation matrices

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.mllib.stat import Statistics

# Prepare vector of df columns for correlation
vector_col = "corr_features"
assembler = VectorAssembler(inputCols=numeric_cols, outputCol=vector_col)
df_vector = assembler.transform(df).select(vector_col)

# Pearson correlation matrix
pearson_matrix = Correlation.corr(df_vector, vector_col, "pearson").collect()[0][0]
pearson_matrix = pearson_matrix.toArray().tolist()
pearson_df = spark.createDataFrame(pearson_matrix, numeric_cols)
display(pearson_df)

# Spearman correlation matrix
spearman_matrix = Correlation.corr(df_vector, vector_col, "spearman").collect()[0][0]
spearman_matrix = spearman_matrix.toArray().tolist()
spearman_df = spark.createDataFrame(spearman_matrix, numeric_cols)
display(spearman_df)

In [0]:
# Convert correlation matrices to Pandas DataFrames, and display as heatmap
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Convert Spark DataFrames to Pandas DataFrames
pearson_pd = pearson_df.toPandas()
spearman_pd = spearman_df.toPandas()
pearson_pd.index = numeric_cols
spearman_pd.index = numeric_cols

# Plot Pearson correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(pearson_pd, annot=True, cmap="YlGnBu")
plt.title("Pearson Correlation Heatmap")
plt.show()

# Plot Spearman correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(spearman_pd, annot=True, cmap="YlGnBu")
plt.title("Spearman Correlation Heatmap")
plt.show()